In [ ]:
import pickle

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from gridworld import Gridworld, random_mdp, build_vocab_with_pad, tokenize_trace, detokenize
from model import GridworldTransformer

In [2]:
# Load the saved dataset and tokens
with open('dataset.pkl', 'rb') as f:
    dataset = pickle.load(f)

with open('tokenized.pkl', 'rb') as f:
    tokenized_data = pickle.load(f)

# Confirm GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Vocab size: {len(tokenized_data['vocab'])}")
print(f"Number of traces: {len(tokenized_data['tokens'])}")

Using device: cuda
Vocab size: 12
Number of traces: 600


In [3]:
vocab = build_vocab_with_pad(max_grid_size=5, actions=['up', 'down', 'left', 'right'])
print(f"New vocab size: {len(vocab)}")
print(f"Pad id: {vocab['<pad>']}")

# Re-tokenize all traces against the new vocab
tokenized = [tokenize_trace(entry['trace'], vocab) for entry in dataset['traces']]
mdp_indices = [entry['mdp_idx'] for entry in dataset['traces']]

New vocab size: 13
Pad id: 0


In [4]:
all_mdp_indices = list(range(10))
rng = np.random.default_rng(0)  # fixed seed for reproducibility
val_mdp_indices = sorted(rng.choice(all_mdp_indices, size=2, replace=False).tolist())
train_mdp_indices = [i for i in all_mdp_indices if i not in val_mdp_indices]

print(f"Train MDPs: {train_mdp_indices}")
print(f"Val MDPs: {val_mdp_indices}")

Train MDPs: [0, 1, 2, 3, 4, 5, 8, 9]
Val MDPs: [6, 7]


In [5]:
class TraceDataset(Dataset):
    def __init__(self, tokenized_traces, mdp_indices, allowed_mdp_indices, context_length, pad_id):
        # filter to traces from allowed MDPs
        self.traces = [
            tokens for tokens, mdp_idx in zip(tokenized_traces, mdp_indices)
            if mdp_idx in allowed_mdp_indices and len(tokens) <= context_length
        ]
        self.context_length = context_length
        self.pad_id = pad_id
    
    def __len__(self):
        return len(self.traces)
    
    def __getitem__(self, idx):
        tokens = self.traces[idx]
        # pad to context_length
        padded = tokens + [self.pad_id] * (self.context_length - len(tokens))
        padded = torch.tensor(padded, dtype=torch.long)
        # input is everything except last position; target is everything except first
        input_ids = padded[:-1]
        target_ids = padded[1:]
        return input_ids, target_ids

In [6]:
context_length = 128
pad_id = vocab['<pad>']

train_dataset = TraceDataset(tokenized, mdp_indices, set(train_mdp_indices), context_length, pad_id)
val_dataset = TraceDataset(tokenized, mdp_indices, set(val_mdp_indices), context_length, pad_id)

print(f"Train traces: {len(train_dataset)}")
print(f"Val traces: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

Train traces: 409
Val traces: 97


In [7]:
input_ids, target_ids = next(iter(train_loader))
print(f"Input shape: {input_ids.shape}")
print(f"Target shape: {target_ids.shape}")
print(f"First input sequence: {input_ids[0]}")
print(f"First target sequence: {target_ids[0]}")

Input shape: torch.Size([32, 127])
Target shape: torch.Size([32, 127])
First input sequence: tensor([ 1,  1,  7, 11,  2,  1, 12,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0])
First target sequence: tensor([ 1,  7, 11,  2,  1, 12,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,

In [8]:
# We need to track this to debug — let's verify
# Reconstruct which MDPs are in train and which trace this is
train_traces_with_mdp = [
    (tokens, mdp_idx) for tokens, mdp_idx in zip(tokenized, mdp_indices)
    if mdp_idx in set(train_mdp_indices) and len(tokens) <= context_length
]
print(f"First train trace's MDP idx: {train_traces_with_mdp[0][1]}")
print(f"That MDP's goal: {dataset['mdps'][train_traces_with_mdp[0][1]]['goal']}")

First train trace's MDP idx: 0
That MDP's goal: (2, 0)


In [9]:
for idx in train_mdp_indices:
    print(f"MDP {idx}: goal={dataset['mdps'][idx]['goal']}")

MDP 0: goal=(2, 0)
MDP 1: goal=(1, 0)
MDP 2: goal=(1, 3)
MDP 3: goal=(2, 2)
MDP 4: goal=(2, 3)
MDP 5: goal=(2, 1)
MDP 8: goal=(3, 2)
MDP 9: goal=(1, 2)


In [10]:
model = GridworldTransformer(
    vocab_size=len(vocab),
    context_length=context_length,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    pad_id=pad_id,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Forward pass on a batch
input_ids, target_ids = next(iter(train_loader))
input_ids = input_ids.to(device)
target_ids = target_ids.to(device)

with torch.no_grad():
    logits = model(input_ids)

print(f"Logits shape: {logits.shape}")  # should be (32, 127, 13)

Total parameters: 76,813
Logits shape: torch.Size([32, 127, 13])


In [11]:
import torch.nn.functional as F

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_tokens = 0
    
    for input_ids, target_ids in loader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        
        # Forward
        logits = model(input_ids)
        # logits: (batch, seq_len, vocab_size)
        # targets: (batch, seq_len)
        # Reshape for cross-entropy
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),  # (batch * seq_len, vocab_size)
            target_ids.reshape(-1)                 # (batch * seq_len,)
        )
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track loss (weighted by non-pad tokens for accurate average)
        non_pad = (target_ids != pad_id).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad
    
    return total_loss / total_tokens

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    for input_ids, target_ids in loader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        
        logits = model(input_ids)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            target_ids.reshape(-1)
        )
        
        non_pad = (target_ids != pad_id).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad
    
    return total_loss / total_tokens

In [12]:
num_epochs = 20

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: train loss = {train_loss:.4f}, val loss = {val_loss:.4f}")

Epoch 1: train loss = 2.2021, val loss = 1.8882
Epoch 2: train loss = 1.7147, val loss = 1.4992
Epoch 3: train loss = 1.3875, val loss = 1.2228
Epoch 4: train loss = 1.1672, val loss = 1.0630
Epoch 5: train loss = 1.0347, val loss = 0.9365
Epoch 6: train loss = 0.9341, val loss = 0.8328
Epoch 7: train loss = 0.8429, val loss = 0.7717
Epoch 8: train loss = 0.7802, val loss = 0.6936
Epoch 9: train loss = 0.7207, val loss = 0.6478
Epoch 10: train loss = 0.6735, val loss = 0.6059
Epoch 11: train loss = 0.6334, val loss = 0.5768
Epoch 12: train loss = 0.6110, val loss = 0.5578
Epoch 13: train loss = 0.5858, val loss = 0.5435
Epoch 14: train loss = 0.5593, val loss = 0.5153
Epoch 15: train loss = 0.5449, val loss = 0.5113
Epoch 16: train loss = 0.5323, val loss = 0.4854
Epoch 17: train loss = 0.5133, val loss = 0.4711
Epoch 18: train loss = 0.5050, val loss = 0.4830
Epoch 19: train loss = 0.4920, val loss = 0.4593
Epoch 20: train loss = 0.4837, val loss = 0.4527


In [13]:
# Check val MDP characteristics vs train MDP characteristics
for label, indices in [('train', train_mdp_indices), ('val', val_mdp_indices)]:
    sizes = [(dataset['mdps'][i]['size_x'], dataset['mdps'][i]['size_y']) for i in indices]
    slips = [dataset['mdps'][i]['slip_prob'] for i in indices]
    print(f"{label} MDPs:")
    for i, (s, sl) in zip(indices, zip(sizes, slips)):
        print(f"  MDP {i}: {s[0]}x{s[1]}, slip={sl:.3f}")
    print(f"  avg slip: {np.mean(slips):.3f}")

train MDPs:
  MDP 0: 5x3, slip=0.042
  MDP 1: 3x5, slip=0.217
  MDP 2: 3x5, slip=0.194
  MDP 3: 4x3, slip=0.049
  MDP 4: 5x5, slip=0.270
  MDP 5: 5x3, slip=0.172
  MDP 8: 5x5, slip=0.249
  MDP 9: 3x5, slip=0.300
  avg slip: 0.186
val MDPs:
  MDP 6: 4x3, slip=0.024
  MDP 7: 4x5, slip=0.297
  avg slip: 0.161


In [14]:
torch.save(model.state_dict(), 'model_v1.pt')

In [15]:
@torch.no_grad()
def per_position_loss(model, loader, device, pad_id, step_length=6):
    """Compute average loss at each position within a step."""
    model.eval()
    
    # accumulate loss and count by position-within-step
    position_losses = {i: 0.0 for i in range(step_length)}
    position_counts = {i: 0 for i in range(step_length)}
    eos_loss = 0.0
    eos_count = 0
    
    for input_ids, target_ids in loader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        
        logits = model(input_ids)  # (batch, seq_len, vocab_size)
        
        # Compute per-token loss without reduction
        loss_per_token = nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            target_ids.reshape(-1),
            reduction='none',
            ignore_index=pad_id
        )
        loss_per_token = loss_per_token.reshape(target_ids.shape)  # (batch, seq_len)
        
        # Mask out padding
        non_pad_mask = (target_ids != pad_id)
        
        for batch_idx in range(target_ids.size(0)):
            seq = target_ids[batch_idx]
            losses = loss_per_token[batch_idx]
            mask = non_pad_mask[batch_idx]
            
            # Find positions of EOS in this sequence
            for pos in range(seq.size(0)):
                if not mask[pos]:
                    continue
                token_id = seq[pos].item()
                # which position-within-step is this?
                pos_in_step = pos % step_length
                
                # Check if this is an EOS token
                if token_id == vocab['<eos>']:
                    eos_loss += losses[pos].item()
                    eos_count += 1
                else:
                    position_losses[pos_in_step] += losses[pos].item()
                    position_counts[pos_in_step] += 1
    
    avg_position_loss = {
        i: position_losses[i] / position_counts[i] if position_counts[i] > 0 else float('nan')
        for i in range(step_length)
    }
    avg_eos_loss = eos_loss / eos_count if eos_count > 0 else float('nan')
    
    return avg_position_loss, avg_eos_loss


position_names = {
    0: 'state_x',
    1: 'state_y',
    2: 'action',
    3: 'reward',
    4: 'next_state_x',
    5: 'next_state_y',
}

print("Per-position loss on train set:")
train_pos_loss, train_eos = per_position_loss(model, train_loader, device, pad_id)
for i, name in position_names.items():
    print(f"  Position {i} ({name}): {train_pos_loss[i]:.4f}")
print(f"  EOS: {train_eos:.4f}")

print("\nPer-position loss on val set:")
val_pos_loss, val_eos = per_position_loss(model, val_loader, device, pad_id)
for i, name in position_names.items():
    print(f"  Position {i} ({name}): {val_pos_loss[i]:.4f}")
print(f"  EOS: {val_eos:.4f}")

Per-position loss on train set:
  Position 0 (state_x): 0.1218
  Position 1 (state_y): 1.0129
  Position 2 (action): 0.3600
  Position 3 (reward): 0.5070
  Position 4 (next_state_x): 0.4866
  Position 5 (next_state_y): 0.1304
  EOS: 0.0255

Per-position loss on val set:
  Position 0 (state_x): 0.1298
  Position 1 (state_y): 0.9630
  Position 2 (action): 0.3493
  Position 3 (reward): 0.6931
  Position 4 (next_state_x): 0.4323
  Position 5 (next_state_y): 0.1441
  EOS: 0.1720


In [16]:
class TraceDatasetByPolicy(Dataset):
    """Same as TraceDataset but also filters by policy type."""
    def __init__(self, dataset, tokenized, mdp_indices, allowed_mdp_indices,
                 allowed_policy_types, context_length, pad_id):
        # Build a mask of which traces to include
        self.traces = []
        for tokens, entry in zip(tokenized, dataset['traces']):
            if (entry['mdp_idx'] in allowed_mdp_indices 
                and entry['policy_type'] in allowed_policy_types
                and len(tokens) <= context_length):
                self.traces.append(tokens)
        self.context_length = context_length
        self.pad_id = pad_id
    
    def __len__(self):
        return len(self.traces)
    
    def __getitem__(self, idx):
        tokens = self.traces[idx]
        padded = tokens + [self.pad_id] * (self.context_length - len(tokens))
        padded = torch.tensor(padded, dtype=torch.long)
        return padded[:-1], padded[1:]


# Build a loader per policy type, on the train set
policy_loaders = {}
for policy_type in ['optimal', 'eps_greedy_01', 'random']:
    ds = TraceDatasetByPolicy(
        dataset, tokenized, mdp_indices, 
        allowed_mdp_indices=set(train_mdp_indices),
        allowed_policy_types={policy_type},
        context_length=context_length,
        pad_id=pad_id,
    )
    policy_loaders[policy_type] = DataLoader(ds, batch_size=32, shuffle=False)
    print(f"{policy_type}: {len(ds)} traces")

print()
for policy_type, loader in policy_loaders.items():
    pos_loss, eos = per_position_loss(model, loader, device, pad_id)
    print(f"{policy_type}:")
    for i, name in position_names.items():
        print(f"  Position {i} ({name}): {pos_loss[i]:.4f}")
    print(f"  EOS: {eos:.4f}")
    print()

optimal: 160 traces
eps_greedy_01: 160 traces
random: 89 traces

optimal:
  Position 0 (state_x): 0.0672
  Position 1 (state_y): 0.5956
  Position 2 (action): 0.3955
  Position 3 (reward): 0.4634
  Position 4 (next_state_x): 0.4048
  Position 5 (next_state_y): 0.0765
  EOS: 0.0229

eps_greedy_01:
  Position 0 (state_x): 0.0801
  Position 1 (state_y): 0.8136
  Position 2 (action): 0.3949
  Position 3 (reward): 0.4533
  Position 4 (next_state_x): 0.4598
  Position 5 (next_state_y): 0.0660
  EOS: 0.0149

random:
  Position 0 (state_x): 0.2043
  Position 1 (state_y): 1.5381
  Position 2 (action): 0.2995
  Position 3 (reward): 0.5910
  Position 4 (next_state_x): 0.5789
  Position 5 (next_state_y): 0.2195
  EOS: 0.0492



In [17]:
import torch.nn.functional as F

@torch.no_grad()
def model_predict_next_state(model, mdp_idx, state, action, reward_token_id, vocab, device, mdp_metadata):
    """
    Use the model to predict the next-state distribution given (state, action).
    Returns a dict {(x, y): probability}.
    
    We construct a minimal context: [state_x, state_y, action, reward].
    The model's predictions at positions 4 and 5 give us next_state_x and next_state_y.
    Joint distribution = P(next_x) * P(next_y | next_x).
    """
    model.eval()
    
    # Build the context tokens
    context = [
        vocab[f'coord_{state[0]}'],
        vocab[f'coord_{state[1]}'],
        vocab[f'action_{action}'],
        reward_token_id,  # we provide both reward options and average
    ]
    
    # Pad to context_length
    pad_id = vocab['<pad>']
    context_length = model.context_length
    padded = context + [pad_id] * (context_length - len(context) - 1)  # -1 for shift
    input_ids = torch.tensor([padded[:context_length-1]], dtype=torch.long).to(device)
    
    logits = model(input_ids)  # (1, seq_len, vocab_size)
    
    # Position 3 in the input (the reward we filled in) means logits at position 3 predict next_x
    # Wait — logits at position i predict input[i+1]. So for context [s_x, s_y, a, r, ?, ?]
    # logits[0] predicts input[1] (s_y)
    # logits[1] predicts input[2] (a)
    # logits[2] predicts input[3] (r)
    # logits[3] predicts input[4] (next_x)  ← this is what we want
    # logits[4] predicts input[5] (next_y)  ← but we need to feed in next_x first
    
    # P(next_x | state, action, reward)
    next_x_logits = logits[0, 3, :]  # (vocab_size,)
    next_x_probs = F.softmax(next_x_logits, dim=-1)
    
    # For each possible next_x, compute P(next_y | state, action, reward, next_x) and combine
    grid_size = max(mdp_metadata['size_x'], mdp_metadata['size_y'])
    coord_token_ids = [vocab[f'coord_{i}'] for i in range(grid_size)]
    
    distribution = {}
    for x in range(mdp_metadata['size_x']):
        x_token = vocab[f'coord_{x}']
        p_x = next_x_probs[x_token].item()
        if p_x < 1e-6:
            continue
        
        # Now feed in next_x and ask for next_y
        extended_context = context + [x_token]
        padded_ext = extended_context + [pad_id] * (context_length - len(extended_context) - 1)
        input_ids_ext = torch.tensor([padded_ext[:context_length-1]], dtype=torch.long).to(device)
        logits_ext = model(input_ids_ext)
        next_y_logits = logits_ext[0, 4, :]
        next_y_probs = F.softmax(next_y_logits, dim=-1)
        
        for y in range(mdp_metadata['size_y']):
            y_token = vocab[f'coord_{y}']
            p_y_given_x = next_y_probs[y_token].item()
            joint_p = p_x * p_y_given_x
            if joint_p > 1e-6:
                distribution[(x, y)] = joint_p
    
    # Renormalize (truncated by ignoring tiny probabilities and out-of-grid coords)
    total = sum(distribution.values())
    if total > 0:
        distribution = {k: v/total for k, v in distribution.items()}
    
    return distribution

In [18]:
from collections import defaultdict, Counter
empirical = defaultdict(Counter)
for entry in dataset['traces']:
    mdp_idx_e = entry['mdp_idx']
    for step in entry['trace']:
        state_e, action_e, reward_e, next_state_e = step
        key = (mdp_idx_e, tuple(state_e), action_e)
        empirical[key][tuple(next_state_e)] += 1

In [19]:
# Pick MDP 9 from earlier (high slip, well-supported)
mdp_idx = 9
mdp_meta = dataset['mdps'][mdp_idx]
mdp = random_mdp(seed=mdp_meta['mdp_seed'])

state = (0, 0)
action = 'right'  # had 74 observations earlier
reward_token = vocab['reward_step']  # not goal

print(f"MDP {mdp_idx}: {mdp_meta['size_x']}x{mdp_meta['size_y']}, goal={mdp_meta['goal']}, slip={mdp_meta['slip_prob']:.3f}")
print(f"Probing transition: state={state}, action={action}\n")

model_dist = model_predict_next_state(model, mdp_idx, state, action, reward_token, vocab, device, mdp_meta)
true_dist = mdp.stochastic_transitions(state, action)
emp_counts = empirical[(mdp_idx, state, action)]
total = sum(emp_counts.values())
emp_dist = {s: c/total for s, c in emp_counts.items()} if total > 0 else {}

print(f"True:      {dict(sorted(true_dist.items()))}")
print(f"Empirical: {dict(sorted(emp_dist.items()))}  (n={total})")
print(f"Model:     {dict(sorted(model_dist.items()))}")

# Compute TV distances
def tv(p, q):
    keys = set(p.keys()) | set(q.keys())
    return 0.5 * sum(abs(p.get(k, 0) - q.get(k, 0)) for k in keys)

print(f"\nTV(true, empirical) = {tv(true_dist, emp_dist):.4f}")
print(f"TV(true, model)     = {tv(true_dist, model_dist):.4f}")
print(f"TV(model, empirical) = {tv(model_dist, emp_dist):.4f}")

MDP 9: 3x5, goal=(1, 2), slip=0.300
Probing transition: state=(0, 0), action=right

True:      {(0, 0): 0.1999522533112331, (0, 1): 0.7000716200331504, (1, 0): 0.09997612665561655}
Empirical: {(0, 0): 0.12162162162162163, (0, 1): 0.7297297297297297, (1, 0): 0.14864864864864866}  (n=74)
Model:     {(0, 0): 0.18078227002624123, (0, 1): 0.7575795111915018, (0, 2): 0.00028964509094207364, (0, 3): 0.00012683220324511012, (0, 4): 0.00016342707694114682, (1, 0): 0.06036838120282875, (1, 1): 0.0004683548677829712, (1, 2): 1.2229242861429917e-05, (1, 3): 1.9930477151986697e-06, (1, 4): 3.6287114104753906e-06, (2, 0): 0.00012161989255371883, (2, 1): 8.210744597607397e-05}

TV(true, empirical) = 0.0783
TV(true, model)     = 0.0588
TV(model, empirical) = 0.0883


In [20]:
results = []

for entry_mdp_idx, mdp_meta in enumerate(dataset['mdps']):
    mdp = random_mdp(seed=mdp_meta['mdp_seed'])
    
    for state in mdp.states:
        if state in mdp.terminal_states:
            continue
        for action in mdp.actions:
            counts = empirical[(entry_mdp_idx, state, action)]
            total = sum(counts.values())
            if total < 30:
                continue
            
            true_dist = mdp.stochastic_transitions(state, action)
            emp_dist = {s: c/total for s, c in counts.items()}
            model_dist = model_predict_next_state(
                model, entry_mdp_idx, state, action, 
                vocab['reward_step'], vocab, device, mdp_meta
            )
            
            results.append({
                'mdp_idx': entry_mdp_idx,
                'state': state,
                'action': action,
                'n': total,
                'tv_true_emp': tv(true_dist, emp_dist),
                'tv_true_model': tv(true_dist, model_dist),
                'tv_emp_model': tv(emp_dist, model_dist),
                'is_train_mdp': entry_mdp_idx in train_mdp_indices,
            })

# Aggregate
import numpy as np

train_results = [r for r in results if r['is_train_mdp']]
val_results = [r for r in results if not r['is_train_mdp']]

print(f"Total well-supported transitions probed: {len(results)}")
print(f"  Train MDPs: {len(train_results)}")
print(f"  Val MDPs: {len(val_results)}")
print()

for label, rs in [('Train MDPs', train_results), ('Val MDPs', val_results), ('All', results)]:
    if not rs:
        continue
    print(f"{label}:")
    print(f"  Mean TV(true, empirical): {np.mean([r['tv_true_emp'] for r in rs]):.4f}")
    print(f"  Mean TV(true, model):     {np.mean([r['tv_true_model'] for r in rs]):.4f}")
    print(f"  Mean TV(empirical, model): {np.mean([r['tv_emp_model'] for r in rs]):.4f}")
    print()

Total well-supported transitions probed: 53
  Train MDPs: 38
  Val MDPs: 15

Train MDPs:
  Mean TV(true, empirical): 0.0590
  Mean TV(true, model):     0.5529
  Mean TV(empirical, model): 0.5621

Val MDPs:
  Mean TV(true, empirical): 0.0286
  Mean TV(true, model):     0.6598
  Mean TV(empirical, model): 0.6662

All:
  Mean TV(true, empirical): 0.0504
  Mean TV(true, model):     0.5832
  Mean TV(empirical, model): 0.5916



In [21]:
# Print the TV distance for each probed (state, action) pair, sorted by state
sorted_results = sorted(results, key=lambda r: (r['mdp_idx'], r['state'], r['action']))
print(f"{'mdp':>4} {'state':>8} {'action':>8} {'n':>4} {'TV(t,e)':>8} {'TV(t,m)':>8}")
for r in sorted_results[:25]:
    print(f"{r['mdp_idx']:>4} {str(r['state']):>8} {r['action']:>8} {r['n']:>4} {r['tv_true_emp']:>8.4f} {r['tv_true_model']:>8.4f}")

 mdp    state   action    n  TV(t,e)  TV(t,m)
   0   (0, 0)     down   63   0.0199   0.1513
   0   (0, 0)     left   37   0.0533   0.0467
   0   (0, 0)       up   34   0.0449   0.0869
   0   (1, 0)     down   48   0.0139   0.9555
   1   (0, 0)     down   62   0.0477   0.0302
   2   (0, 0)     left   36   0.0743   0.0551
   2   (0, 0)    right   70   0.0354   0.0529
   2   (0, 0)       up   32   0.0646   0.0458
   2   (0, 1)    right   65   0.0676   0.8584
   2   (0, 2)    right   52   0.0948   0.9346
   2   (0, 3)     down   32   0.1312   0.9354
   2   (1, 0)    right   32   0.0374   0.9102
   2   (1, 1)    right   37   0.1141   0.8727
   3   (0, 0)    right   72   0.0230   0.1934
   3   (0, 1)     down   60   0.0179   0.9359
   3   (1, 1)    right   56   0.0163   0.9673
   3   (1, 2)     down   49   0.0163   0.9788
   4   (0, 0)     down   36   0.0480   0.0837
   4   (0, 0)     left   46   0.0838   0.1062
   4   (0, 0)    right   91   0.0705   0.0297
   4   (0, 0)       up   45   0.02

In [22]:
@torch.no_grad()
def model_predict_with_real_context(model, mdp_idx, state, action, vocab, device, mdp_meta, dataset, tokenized, mdp_indices):
    """
    Probe the model using a real trace prefix as context.
    Find any trace from this MDP that visits 'state', use the prefix up to that visit,
    then append (action, reward) and ask for next_state.
    """
    model.eval()
    pad_id = vocab['<pad>']
    context_length = model.context_length
    
    # Find a trace from this MDP that visits state
    target_state_x = vocab[f'coord_{state[0]}']
    target_state_y = vocab[f'coord_{state[1]}']
    
    prefix_tokens = None
    for tokens, mdp_i in zip(tokenized, mdp_indices):
        if mdp_i != mdp_idx:
            continue
        # Walk through the trace looking for our state at the start of a step
        # Steps start at positions 0, 6, 12, ...
        for step_start in range(0, len(tokens) - 6, 6):
            if tokens[step_start] == target_state_x and tokens[step_start + 1] == target_state_y:
                # Found it. Use the prefix up to this state, then append our action.
                prefix_tokens = tokens[:step_start + 2] + [vocab[f'action_{action}']]
                break
        if prefix_tokens is not None:
            break
    
    if prefix_tokens is None:
        return None  # state never visited in this MDP's traces
    
    # Pad
    if len(prefix_tokens) >= context_length:
        return None  # context too long
    
    padded = prefix_tokens + [pad_id] * (context_length - 1 - len(prefix_tokens))
    input_ids = torch.tensor([padded[:context_length-1]], dtype=torch.long).to(device)
    
    logits = model(input_ids)
    
    # logits[i] predicts position i+1
    # The action we just placed is at position len(prefix_tokens) - 1
    # So logits at len(prefix_tokens) - 1 predicts the next token (reward)
    # We want next_state, which means we need to feed reward too then look further
    
    # Predict reward
    pos = len(prefix_tokens) - 1
    reward_logits = logits[0, pos, :]
    reward_probs = F.softmax(reward_logits, dim=-1)
    
    # Marginalize over reward outcomes
    distribution = {}
    for reward_token_name in ['reward_step', 'reward_goal']:
        reward_token_id = vocab[reward_token_name]
        p_r = reward_probs[reward_token_id].item()
        if p_r < 1e-6:
            continue
        
        # Extend with this reward and predict next_x
        ext_prefix = prefix_tokens + [reward_token_id]
        padded_r = ext_prefix + [pad_id] * (context_length - 1 - len(ext_prefix))
        input_r = torch.tensor([padded_r[:context_length-1]], dtype=torch.long).to(device)
        logits_r = model(input_r)
        next_x_logits = logits_r[0, len(ext_prefix) - 1, :]
        next_x_probs = F.softmax(next_x_logits, dim=-1)
        
        for x in range(mdp_meta['size_x']):
            x_token = vocab[f'coord_{x}']
            p_x = next_x_probs[x_token].item()
            if p_x < 1e-6:
                continue
            
            # Extend further with this next_x and predict next_y
            ext_prefix_x = ext_prefix + [x_token]
            padded_x = ext_prefix_x + [pad_id] * (context_length - 1 - len(ext_prefix_x))
            input_x = torch.tensor([padded_x[:context_length-1]], dtype=torch.long).to(device)
            logits_x = model(input_x)
            next_y_logits = logits_x[0, len(ext_prefix_x) - 1, :]
            next_y_probs = F.softmax(next_y_logits, dim=-1)
            
            for y in range(mdp_meta['size_y']):
                y_token = vocab[f'coord_{y}']
                p_y = next_y_probs[y_token].item()
                joint_p = p_r * p_x * p_y
                if joint_p > 1e-6:
                    key = (x, y)
                    distribution[key] = distribution.get(key, 0) + joint_p
    
    # Renormalize
    total = sum(distribution.values())
    if total > 0:
        distribution = {k: v/total for k, v in distribution.items()}
    
    return distribution

In [23]:
results_v2 = []

for entry_mdp_idx, mdp_meta in enumerate(dataset['mdps']):
    mdp = random_mdp(seed=mdp_meta['mdp_seed'])
    
    for state in mdp.states:
        if state in mdp.terminal_states:
            continue
        for action in mdp.actions:
            counts = empirical[(entry_mdp_idx, state, action)]
            total = sum(counts.values())
            if total < 30:
                continue
            
            true_dist = mdp.stochastic_transitions(state, action)
            emp_dist = {s: c/total for s, c in counts.items()}
            model_dist = model_predict_with_real_context(
                model, entry_mdp_idx, state, action, 
                vocab, device, mdp_meta, dataset, tokenized, mdp_indices
            )
            
            if model_dist is None:
                continue
            
            results_v2.append({
                'mdp_idx': entry_mdp_idx,
                'state': state,
                'action': action,
                'n': total,
                'tv_true_emp': tv(true_dist, emp_dist),
                'tv_true_model': tv(true_dist, model_dist),
                'tv_emp_model': tv(emp_dist, model_dist),
                'is_train_mdp': entry_mdp_idx in train_mdp_indices,
            })

train_results = [r for r in results_v2 if r['is_train_mdp']]
val_results = [r for r in results_v2 if not r['is_train_mdp']]

print(f"Total: {len(results_v2)}")
for label, rs in [('Train MDPs', train_results), ('Val MDPs', val_results), ('All', results_v2)]:
    if not rs:
        continue
    print(f"{label}:")
    print(f"  Mean TV(true, empirical): {np.mean([r['tv_true_emp'] for r in rs]):.4f}")
    print(f"  Mean TV(true, model):     {np.mean([r['tv_true_model'] for r in rs]):.4f}")
    print(f"  Mean TV(empirical, model): {np.mean([r['tv_emp_model'] for r in rs]):.4f}")

Total: 53
Train MDPs:
  Mean TV(true, empirical): 0.0590
  Mean TV(true, model):     0.1649
  Mean TV(empirical, model): 0.1732
Val MDPs:
  Mean TV(true, empirical): 0.0286
  Mean TV(true, model):     0.2882
  Mean TV(empirical, model): 0.3023
All:
  Mean TV(true, empirical): 0.0504
  Mean TV(true, model):     0.1998
  Mean TV(empirical, model): 0.2097


In [24]:
# Train the larger model
model_large = GridworldTransformer(
    vocab_size=len(vocab),
    context_length=context_length,
    d_model=128,
    nhead=8,
    num_layers=4,
    dim_feedforward=256,
    pad_id=pad_id,
).to(device)

total_params_large = sum(p.numel() for p in model_large.parameters())
print(f"Large model parameters: {total_params_large:,}")

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer_large = torch.optim.AdamW(model_large.parameters(), lr=1e-3)

num_epochs = 20

train_losses_large = []
val_losses_large = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model_large, train_loader, criterion, optimizer_large, device)
    val_loss = evaluate(model_large, val_loader, criterion, device)
    train_losses_large.append(train_loss)
    val_losses_large.append(val_loss)
    print(f"Epoch {epoch+1}: train loss = {train_loss:.4f}, val loss = {val_loss:.4f}")

# Save it
torch.save(model_large.state_dict(), 'model_large_v1.pt')

Large model parameters: 549,645
Epoch 1: train loss = 1.6982, val loss = 1.2398
Epoch 2: train loss = 1.0638, val loss = 0.8823
Epoch 3: train loss = 0.8204, val loss = 0.7164
Epoch 4: train loss = 0.6852, val loss = 0.6436
Epoch 5: train loss = 0.6150, val loss = 0.5859
Epoch 6: train loss = 0.5525, val loss = 0.5299
Epoch 7: train loss = 0.5195, val loss = 0.4868
Epoch 8: train loss = 0.4858, val loss = 0.4629
Epoch 9: train loss = 0.4533, val loss = 0.4452
Epoch 10: train loss = 0.4409, val loss = 0.4839
Epoch 11: train loss = 0.4232, val loss = 0.4344
Epoch 12: train loss = 0.4003, val loss = 0.4289
Epoch 13: train loss = 0.3880, val loss = 0.4418
Epoch 14: train loss = 0.3774, val loss = 0.4194
Epoch 15: train loss = 0.3601, val loss = 0.4396
Epoch 16: train loss = 0.3486, val loss = 0.4180
Epoch 17: train loss = 0.3416, val loss = 0.4441
Epoch 18: train loss = 0.3375, val loss = 0.4365
Epoch 19: train loss = 0.3230, val loss = 0.4452
Epoch 20: train loss = 0.3178, val loss = 0.43

In [25]:
torch.save(model_large.state_dict(), 'model_large_v1.pt')

In [26]:
results_large = []

for entry_mdp_idx, mdp_meta in enumerate(dataset['mdps']):
    mdp = random_mdp(seed=mdp_meta['mdp_seed'])
    
    for state in mdp.states:
        if state in mdp.terminal_states:
            continue
        for action in mdp.actions:
            counts = empirical[(entry_mdp_idx, state, action)]
            total = sum(counts.values())
            if total < 30:
                continue
            
            true_dist = mdp.stochastic_transitions(state, action)
            emp_dist = {s: c/total for s, c in counts.items()}
            model_dist = model_predict_with_real_context(
                model_large, entry_mdp_idx, state, action, 
                vocab, device, mdp_meta, dataset, tokenized, mdp_indices
            )
            
            if model_dist is None:
                continue
            
            results_large.append({
                'mdp_idx': entry_mdp_idx,
                'state': state,
                'action': action,
                'n': total,
                'tv_true_emp': tv(true_dist, emp_dist),
                'tv_true_model': tv(true_dist, model_dist),
                'tv_emp_model': tv(emp_dist, model_dist),
                'is_train_mdp': entry_mdp_idx in train_mdp_indices,
            })

train_results_large = [r for r in results_large if r['is_train_mdp']]
val_results_large = [r for r in results_large if not r['is_train_mdp']]

print("LARGE MODEL")
for label, rs in [('Train MDPs', train_results_large), ('Val MDPs', val_results_large), ('All', results_large)]:
    if not rs:
        continue
    print(f"{label}:")
    print(f"  Mean TV(true, model): {np.mean([r['tv_true_model'] for r in rs]):.4f}")

# Compare to small
print("\nSMALL MODEL (from earlier)")
for label, rs in [('Train MDPs', train_results), ('Val MDPs', val_results)]:
    print(f"{label}:")
    print(f"  Mean TV(true, model): {np.mean([r['tv_true_model'] for r in rs]):.4f}")

LARGE MODEL
Train MDPs:
  Mean TV(true, model): 0.1252
Val MDPs:
  Mean TV(true, model): 0.1760
All:
  Mean TV(true, model): 0.1396

SMALL MODEL (from earlier)
Train MDPs:
  Mean TV(true, model): 0.1649
Val MDPs:
  Mean TV(true, model): 0.2882


In [27]:
# Load all checkpoints
checkpoints = {
    'small_seed0': 'model_v1.pt',  # your original
    'small_seed1': 'model_small_seed1.pt',
    'small_seed2': 'model_small_seed2.pt',
    'large_seed0': 'model_large_v1.pt',  # your original
    'large_seed1': 'model_large_seed1.pt',
    'large_seed2': 'model_large_seed2.pt',
}

def make_model(size):
    if size == 'small':
        return GridworldTransformer(
            vocab_size=len(vocab), context_length=context_length,
            d_model=64, nhead=4, num_layers=2, dim_feedforward=128, pad_id=pad_id
        ).to(device)
    else:
        return GridworldTransformer(
            vocab_size=len(vocab), context_length=context_length,
            d_model=128, nhead=8, num_layers=4, dim_feedforward=256, pad_id=pad_id
        ).to(device)

all_results = {}

for name, path in checkpoints.items():
    size = 'small' if 'small' in name else 'large'
    m = make_model(size)
    m.load_state_dict(torch.load(path))
    m.eval()
    
    print(f"Probing {name}...")
    results = []
    for entry_mdp_idx, mdp_meta in enumerate(dataset['mdps']):
        mdp = random_mdp(seed=mdp_meta['mdp_seed'])
        for state in mdp.states:
            if state in mdp.terminal_states:
                continue
            for action in mdp.actions:
                counts = empirical[(entry_mdp_idx, state, action)]
                total = sum(counts.values())
                if total < 30:
                    continue
                
                true_dist = mdp.stochastic_transitions(state, action)
                emp_dist = {s: c/total for s, c in counts.items()}
                model_dist = model_predict_with_real_context(
                    m, entry_mdp_idx, state, action, 
                    vocab, device, mdp_meta, dataset, tokenized, mdp_indices
                )
                if model_dist is None:
                    continue
                
                results.append({
                    'mdp_idx': entry_mdp_idx,
                    'is_train_mdp': entry_mdp_idx in train_mdp_indices,
                    'tv_true_model': tv(true_dist, model_dist),
                })
    all_results[name] = results

Probing small_seed0...


/tmp/ipykernel_1665393/808183995.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m.load_state_dict(torch.load(path))


Probing small_seed1...
Probing small_seed2...
Probing large_seed0...
Probing large_seed1...
Probing large_seed2...


In [28]:
print(f"{'Model':<20} {'Train TV (mean±std)':<25} {'Val TV (mean±std)':<25}")
print("-" * 70)

# Group by architecture
for arch in ['small', 'large']:
    train_tvs_per_seed = []
    val_tvs_per_seed = []
    
    for seed in [0, 1, 2]:
        name = f'{arch}_seed{seed}'
        results = all_results[name]
        train_results = [r['tv_true_model'] for r in results if r['is_train_mdp']]
        val_results = [r['tv_true_model'] for r in results if not r['is_train_mdp']]
        train_tvs_per_seed.append(np.mean(train_results))
        val_tvs_per_seed.append(np.mean(val_results))
        print(f"{name:<20} train={np.mean(train_results):.4f} val={np.mean(val_results):.4f}")
    
    train_mean = np.mean(train_tvs_per_seed)
    train_std = np.std(train_tvs_per_seed)
    val_mean = np.mean(val_tvs_per_seed)
    val_std = np.std(val_tvs_per_seed)
    print(f"{arch.upper():<20} train={train_mean:.4f}±{train_std:.4f}  val={val_mean:.4f}±{val_std:.4f}")
    print()

Model                Train TV (mean±std)       Val TV (mean±std)        
----------------------------------------------------------------------
small_seed0          train=0.1649 val=0.2882
small_seed1          train=0.1722 val=0.2984
small_seed2          train=0.1864 val=0.2697
SMALL                train=0.1745±0.0089  val=0.2854±0.0119

large_seed0          train=0.1252 val=0.1760
large_seed1          train=0.1203 val=0.2095
large_seed2          train=0.1237 val=0.1981
LARGE                train=0.1231±0.0020  val=0.1945±0.0139

